In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL

cwd = Path.cwd()
print(cwd)
root=cwd/'institutional-roi-analysis'
pd.set_option("display.max_columns",None)
display(root)

C:\Users\sebas\PycharmProjects\Git\Seb_branch


WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [2]:
tdf = pd.read_parquet(root/"data"/"raw"/"scorecard"/"national_scorecard.parquet")
display(tdf.head())
display(tdf.info())

,code,title,unit_id,distance,school.type,credential.level,earnings.1_yr.overall_median_earnings,earnings.1_yr.working_not_enrolled.overall_count,earnings.4_yr.overall_median_earnings,earnings.4_yr.working_not_enrolled.overall_count,earnings.5_yr.overall_median_earnings,earnings.5_yr.working_not_enrolled.overall_count,id,school.name,school.state,location.lat,location.lon,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type,latest.admissions.sat_scores.average.overall,latest.admissions.act_scores.midpoint.cumulative,latest.student.grad_students
0,0305,Forestry.,100654,1,Public,3,NaN,NaN,64749,16,NaN,NaN,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
1,1002,Audiovisual Communications Technologies/Techni...,100654,1,Public,3,28938.0,31.0,42272,37,NaN,NaN,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
2,1101,"Computer and Information Sciences, General.",100654,1,Public,3,63900.0,29.0,88490,39,85218.0,27.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
3,1312,Teacher Education and Professional Development...,100654,2,Public,5,56295.0,21.0,60412,24,69062.0,18.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
4,1410,"Electrical, Electronics, and Communications En...",100654,1,Public,3,72241.0,40.0,98045,43,90409.0,29.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55930 entries, 0 to 55929
Data columns (total 28 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   code                                              55930 non-null  object 
 1   title                                             55930 non-null  object 
 2   unit_id                                           55930 non-null  int64  
 3   distance                                          55930 non-null  int64  
 4   school.type                                       55930 non-null  object 
 5   credential.level                                  55930 non-null  int64  
 6   earnings.1_yr.overall_median_earnings             46006 non-null  float64
 7   earnings.1_yr.working_not_enrolled.overall_count  46006 non-null  float64
 8   earnings.4_yr.overall_median_earnings             55930 non-null  int64  
 9   earnings.4_yr.wor

None

# Why so many missing Admission Rates?

In [3]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

np.float64(0.3301)

In [4]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

np.float64(0.005)

Approximately 40% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [5]:
df = df.drop(columns="id")
df = clean(df)

In [6]:
df["selectivity_bucket"] = pd.cut(
    df["admission_rate_overall"],
    bins=[0, 0.3, 0.7, 1],
    labels=["elite", "mid", "open"]
)

In [7]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55930 entries, 0 to 55929
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   code                            55930 non-null  string  
 1   title                           55930 non-null  string  
 2   unit_id                         55930 non-null  string  
 3   distance                        55930 non-null  int64   
 4   school_type                     55930 non-null  string  
 5   credential_level                55930 non-null  int64   
 6   1_yr_median_earnings            46006 non-null  float64 
 7   1_yr_working_count              46006 non-null  float64 
 8   4_yr_median_earnings            55930 non-null  int64   
 9   4_yr_working_count              55930 non-null  int64   
 10  5_yr_median_earnings            41343 non-null  float64 
 11  5_yr_working_count              41343 non-null  float64 
 12  school_name       

None

In [8]:
df.head()

,code,title,unit_id,distance,school_type,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,school_name,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket
0,0305,Forestry.,100654,1,Public,3,NaN,NaN,64749,16,NaN,NaN,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
1,1002,Audiovisual Communications Technologies/Techni...,100654,1,Public,3,28938.0,31.0,42272,37,NaN,NaN,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
2,1101,"Computer and Information Sciences, General.",100654,1,Public,3,63900.0,29.0,88490,39,85218.0,27.0,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
3,1312,Teacher Education and Professional Development...,100654,2,Public,5,56295.0,21.0,60412,24,69062.0,18.0,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
4,1410,"Electrical, Electronics, and Communications En...",100654,1,Public,3,72241.0,40.0,98045,43,90409.0,29.0,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid


In [9]:
save(df,file_type="scorecard",clean=0,file_name="national_preprocessed_scorecard_programs")

In [10]:
display("Relevant numeric variables statistics:", df.select_dtypes(exclude=object).describe())
display("Missing values per column:", df.isna().sum())
display("Correlation matrix:", df.corr(numeric_only=True))

'Relevant numeric variables statistics:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students
count,55930.000000,55930.000000,46006.000000,46006.000000,55930.000000,55930.000000,41343.000000,41343.000000,55930.000000,55930.000000,55930.000000,51620.000000,37469.0,55566.000000,53574.0,54911.000000,55566.000000,55930.000000,28424.000000,25623.000000,39457.000000
mean,1.298301,3.423368,49347.405447,110.133591,65571.535759,101.731164,63604.452483,108.088358,37.955487,-89.541457,18.530753,11.347792,0.720972,42843.610319,0.633126,1.685819,23.105226,1.018309,1209.583697,25.467275,4507.224345
std,0.739933,5.726003,24051.517980,300.209665,27720.152721,280.310977,28396.672776,278.890877,5.436187,15.649505,9.133195,4.841543,0.2318,22801.900981,0.176964,0.464193,3.379662,0.228855,143.918529,4.015281,6172.442970
min,0.000000,1.000000,4506.000000,16.000000,8305.000000,16.000000,7826.000000,16.000000,-14.322636,-170.742774,11.000000,1.000000,0.0,0.000000,0.114961,1.000000,17.000000,1.000000,720.000000,14.000000,1.000000
25%,1.000000,2.000000,32459.250000,28.000000,47566.000000,25.000000,45549.000000,28.000000,34.152076,-96.581077,11.000000,9.000000,0.6154,24244.000000,0.486853,1.000000,21.000000,1.000000,1099.000000,23.000000,725.000000
50%,1.000000,3.000000,44141.500000,47.000000,59562.500000,43.000000,57882.000000,47.000000,39.328977,-85.533519,13.000000,13.000000,0.7841,37667.500000,0.62919,2.000000,22.000000,1.000000,1190.000000,25.000000,2264.000000
75%,1.000000,3.000000,61827.750000,98.000000,77911.750000,91.000000,75760.500000,97.000000,41.703058,-78.157433,21.000000,15.000000,0.8901,58609.000000,0.777258,2.000000,25.000000,1.000000,1297.000000,28.000000,5711.000000
max,3.000000,99.000000,272682.000000,11263.000000,336392.000000,9665.000000,384547.000000,10468.000000,64.857560,145.721733,43.000000,18.000000,1.0,179864.000000,0.99674,2.000000,48.000000,5.000000,1560.000000,35.000000,55120.000000


'Missing values per column:'

code                                  0
title                                 0
unit_id                               0
distance                              0
school_type                           0
credential_level                      0
1_yr_median_earnings               9924
1_yr_working_count                 9924
4_yr_median_earnings                  0
4_yr_working_count                    0
5_yr_median_earnings              14587
5_yr_working_count                14587
school_name                           0
school_state                          0
location_lat                          0
location_lon                          0
locale                                0
carnegie_size_setting              4310
admission_rate_overall            18461
median_family_income                364
students_with_pell_grant           2356
open_admissions_policy             1019
age_entry                           364
title_iv_eligibility_type             0
sat_scores_average_overall        27506


'Correlation matrix:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students
distance,1.000000,-0.064231,0.087875,0.045212,0.026595,0.060110,0.011334,0.049668,-0.023778,0.009573,0.027418,-0.029884,0.090494,-0.100275,0.084193,-0.063824,0.246856,0.149040,-0.125451,-0.139590,0.066216
credential_level,-0.064231,1.000000,0.528689,-0.049601,0.127210,-0.022065,0.542649,-0.059269,-0.005253,-0.013301,-0.048364,0.127033,-0.021647,0.075802,-0.081029,0.117613,-0.045270,0.001576,0.025357,0.024428,0.092013
1_yr_median_earnings,0.087875,0.528689,1.000000,0.032196,0.912971,0.012966,0.871953,-0.004042,0.108403,-0.003562,-0.089905,0.216672,-0.185652,0.226013,-0.261235,0.237967,-0.088883,0.020126,0.243557,0.240116,0.163279
1_yr_working_count,0.045212,-0.049601,0.032196,1.000000,0.015639,0.972806,0.020124,0.826064,-0.054290,-0.041894,-0.056637,-0.041627,0.027674,-0.089292,0.078346,-0.092510,0.178741,0.072431,0.023925,0.051733,0.143269
4_yr_median_earnings,0.026595,0.127210,0.912971,0.015639,1.000000,0.013077,0.943206,-0.011333,0.135369,-0.004781,-0.134777,0.336668,-0.277068,0.330410,-0.349991,0.335767,-0.202435,0.008376,0.365682,0.364367,0.217848
4_yr_working_count,0.060110,-0.022065,0.012966,0.972806,0.013077,1.000000,0.013833,0.884454,-0.051139,-0.033560,-0.057614,-0.038005,0.030593,-0.087938,0.079297,-0.089302,0.177095,0.086418,0.016139,0.045802,0.117354
5_yr_median_earnings,0.011334,0.542649,0.871953,0.020124,0.943206,0.013833,1.000000,-0.014781,0.132200,-0.006525,-0.124917,0.342259,-0.284752,0.339342,-0.364824,0.354463,-0.223239,0.002439,0.374018,0.374147,0.209618
5_yr_working_count,0.049668,-0.059269,-0.004042,0.826064,-0.011333,0.884454,-0.014781,1.000000,-0.061587,-0.050599,-0.061436,-0.071796,0.042269,-0.116831,0.112136,-0.122692,0.206762,0.059092,0.008662,0.059848,0.083087
location_lat,-0.023778,-0.005253,0.108403,-0.054290,0.135369,-0.051139,0.132200,-0.061587,1.000000,0.021273,0.108338,0.010766,0.108251,0.359370,-0.365666,0.095307,-0.113198,-0.023495,0.145722,0.151120,-0.041482
location_lon,0.009573,-0.013301,-0.003562,-0.041894,-0.004781,-0.033560,-0.006525,-0.050599,0.021273,1.000000,0.100717,0.006911,-0.107101,0.189199,-0.211509,0.082897,-0.140627,0.032241,0.138953,0.196015,-0.038194


In [11]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split, cross_val_predict
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from xgboost import XGBRegressor

# Reproducibility
RANDOM_STATE = 42

In [ ]:
def run_earnings_model(df, year, model_type="xgb", random_state=42, cv=5):
    year = int(year)
    target = f"{year}_yr_median_earnings"

    other_year_cols = []
    for y in [1, 4, 5]:
        if y != year:
            other_year_cols.extend([
                f"{y}_yr_median_earnings",
                f"{y}_yr_working_count",
            ])

    model_df = df.copy()
    model_df[target] = pd.to_numeric(model_df[target], errors="coerce")
    model_df = model_df[model_df[target].notna() & (model_df[target] > 0)].copy()

    drop_columns = ["title", "school_name"] + other_year_cols
    X = model_df.drop(columns=drop_columns, errors="ignore").copy()
    X = X.drop(columns=[target], errors="ignore")
    y = pd.to_numeric(model_df[target], errors="coerce").copy()
    y_log = np.log(y)

    cat_cols = [
        'code', 'school_type', 'locale', 'carnegie_size_setting',
        'open_admissions_policy', 'title_iv_eligibility_type',
        'credential_level', 'distance', 'selectivity_bucket', 'state'
    ]
    num_cols = [
        'admission_rate_overall', 'location_lat', 'location_lon',
        'median_family_income', 'students_with_pell_grant', 'age_entry',
        'sat_scores_average_overall','act_scores_midpoint_cumulative',
        'grad_students'
    ]

    cat_cols = [c for c in cat_cols if c in X.columns]
    num_cols = [c for c in num_cols if c in X.columns]

    for c in cat_cols:
        X[c] = X[c].astype(str).replace({"nan": "Missing", "<NA>": "Missing"})
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X_train, X_test, y_train_log, y_test_log = train_test_split(
        X, y_log, test_size=0.2, random_state=random_state
    )

    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("to_str", FunctionTransformer(lambda X: X.astype(str))),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ('poly', PolynomialFeatures(degree=2, interaction_only=True))
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ])

    # ── model selection ──────────────────────────────────────────
    if model_type == "xgb":
        estimator = XGBRegressor(random_state=random_state, n_jobs=-1, verbosity=0)
        param_grid = {
            "reg__n_estimators": [200, 300],
            "reg__max_depth": [5, 10],
            "reg__learning_rate": [0.05, 0.1],
            "reg__subsample": [0.8, 1.0],
        }
    elif model_type == "ridge":
        estimator = Ridge()
        param_grid = {
            "reg__alpha": [
                0.001, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0, 2.0
            ]
        }
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("reg", estimator)
    ])

    grid = GridSearchCV(
        pipe, param_grid,
        scoring="neg_mean_absolute_error",
        cv=cv, refit=True, verbose=1, error_score="raise"
    )

    grid.fit(X_train, y_train_log)

    print(f"\n===== {year}-YEAR MODEL ({model_type.upper()}) =====")
    print("Best params:", grid.best_params_)
    print("Best CV MAE (log):", round(-grid.best_score_, 4))

    best_model = grid.best_estimator_
    preds_log = best_model.predict(X_test)
    print("Test MAE (log):", round(mean_absolute_error(y_test_log, preds_log), 4))
    print("Test R² (log):", round(r2_score(y_test_log, preds_log), 4))

    # OOF predictions on full dataset
    cv_preds_log = cross_val_predict(
        best_model, X, y_log, cv=cv, method="predict", n_jobs=1
    )

    # build error_df
    actual_log_col = f"{year}_year_earning_log"
    pred_log_col   = f"{year}_year_pred_log"
    error_log_col  = f"{year}_year_error_log"
    actual_col     = f"{year}_year_earning"
    pred_col       = f"{year}_year_pred"
    error_col      = f"{year}_year_error"

    error_df = X.copy()
    error_df[actual_log_col] = y_log.values
    error_df[pred_log_col]   = cv_preds_log

    cols_to_add = [c for c in ["title", "school_name", "credential_level"] if c in model_df.columns]
    error_df[cols_to_add] = model_df.loc[error_df.index, cols_to_add]
    error_df[error_log_col]  = error_df[actual_log_col] - error_df[pred_log_col]

    error_df[actual_col] = np.exp(error_df[actual_log_col])
    error_df[pred_col]   = np.exp(error_df[pred_log_col])
    error_df[error_col]  = error_df[actual_col] - error_df[pred_col]

    return {
        "year": year,
        "model_type": model_type,
        "target": target,
        "grid": grid,
        "best_model": best_model,
        "X": X,
        "y": y,
        "error_df": error_df,
    }

In [66]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model_name='xgb'

results_1 = run_earnings_model(df, year=1,model_type=model_name, random_state=RANDOM_STATE, cv=cv)
results_4 = run_earnings_model(df, year=4,model_type=model_name, random_state=RANDOM_STATE, cv=cv)
results_5 = run_earnings_model(df, year=5,model_type=model_name, random_state=RANDOM_STATE, cv=cv)

Fitting 5 folds for each of 16 candidates, totalling 80 fits

===== 1-YEAR MODEL (XGB) =====
Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
Best CV MAE (log): 0.1374
Test MAE (log): 0.1363
Test R² (log): 0.829
Fitting 5 folds for each of 16 candidates, totalling 80 fits

===== 4-YEAR MODEL (XGB) =====
Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
Best CV MAE (log): 0.1168
Test MAE (log): 0.1144
Test R² (log): 0.8402
Fitting 5 folds for each of 16 candidates, totalling 80 fits

===== 5-YEAR MODEL (XGB) =====
Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
Best CV MAE (log): 0.1165
Test MAE (log): 0.1135
Test R² (log): 0.8553


In [67]:
df_1=results_1["error_df"]
display(df_1.head())
display(results_1["error_df"].shape)

df_4=results_4["error_df"]
display(df_4.head())
display(results_4["error_df"].shape)

df_5=results_5["error_df"]
display(df_5.head())
display(results_5["error_df"].shape)

,code,unit_id,distance,school_type,credential_level,1_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,1_year_earning_log,1_year_pred_log,title,school_name,1_year_error_log,1_year_earning,1_year_pred,1_year_error
1,1002,100654,1,Public,3,31.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.272911,10.250331,Audiovisual Communications Technologies/Techni...,Alabama A & M University,0.022580,28938.0,28291.902344,646.097656
2,1101,100654,1,Public,3,29.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.065075,10.684943,"Computer and Information Sciences, General.",Alabama A & M University,0.380131,63900.0,43693.000000,20207.000000
3,1312,100654,2,Public,5,21.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.938361,10.814303,Teacher Education and Professional Development...,Alabama A & M University,0.124058,56295.0,49727.000000,6568.000000
4,1410,100654,1,Public,3,40.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.187763,11.122828,"Electrical, Electronics, and Communications En...",Alabama A & M University,0.064936,72241.0,67699.062500,4541.937500
5,1419,100654,1,Public,3,45.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.183782,11.039113,Mechanical Engineering.,Alabama A & M University,0.144669,71954.0,62262.406250,9691.593750


(46006, 29)

,code,unit_id,distance,school_type,credential_level,4_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,4_year_earning_log,4_year_pred_log,title,school_name,4_year_error_log,4_year_earning,4_year_pred,4_year_error
0,0305,100654,1,Public,3,16,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.078274,10.815783,Forestry.,Alabama A & M University,0.262491,64749.0,49800.609375,14948.390625
1,1002,100654,1,Public,3,37,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.651880,10.634857,Audiovisual Communications Technologies/Techni...,Alabama A & M University,0.017023,42272.0,41558.492188,713.507812
2,1101,100654,1,Public,3,39,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.390645,11.036671,"Computer and Information Sciences, General.",Alabama A & M University,0.353974,88490.0,62110.523438,26379.476562
3,1312,100654,2,Public,5,24,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.008943,10.883490,Teacher Education and Professional Development...,Alabama A & M University,0.125453,60412.0,53289.234375,7122.765625
4,1410,100654,1,Public,3,43,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.493182,11.346757,"Electrical, Electronics, and Communications En...",Alabama A & M University,0.146425,98045.0,84690.351562,13354.648438


(55930, 29)

,code,unit_id,distance,school_type,credential_level,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_error_log,5_year_earning,5_year_pred,5_year_error
2,1101,100654,1,Public,3,27.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.352968,11.003503,"Computer and Information Sciences, General.",Alabama A & M University,0.349465,85218.0,60084.242188,25133.757812
3,1312,100654,2,Public,5,18.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.142760,10.903197,Teacher Education and Professional Development...,Alabama A & M University,0.239563,69062.0,54349.855469,14712.144531
4,1410,100654,1,Public,3,29.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.412099,11.241163,"Electrical, Electronics, and Communications En...",Alabama A & M University,0.170936,90409.0,76203.546875,14205.453125
5,1419,100654,1,Public,3,22.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,11.325740,11.239490,Mechanical Engineering.,Alabama A & M University,0.086251,82929.0,76076.109375,6852.890625
8,2401,100654,1,Public,3,30.0,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid,10.749935,10.646960,"Liberal Arts and Sciences, General Studies and...",Alabama A & M University,0.102975,46627.0,42064.535156,4562.464844


(41343, 29)

In [68]:
df_1['1_yr_working_count'].isna().sum()

np.int64(0)

In [69]:
merge_df=df_5.merge(df_4[['4_year_earning','4_year_earning_log','4_year_pred','4_year_pred_log','4_year_error','4_year_error_log',"4_yr_working_count",'code','unit_id','credential_level']],on=['code','unit_id','credential_level'],how="inner")
merge_df=merge_df.merge(df_1[['1_year_earning','1_year_earning_log','1_year_pred','1_year_pred_log','1_year_error','1_year_error_log',"1_yr_working_count",'code','unit_id','credential_level']],on=['code','unit_id','credential_level'],how="inner")
merge_df.shape

(36653, 43)

In [70]:
error_df=merge_df.copy()
valid_codes = (
    error_df.groupby(["code","credential_level"])["school_name"]
    .nunique()
)

valid_codes = valid_codes[valid_codes >= 3].index
valid_codes

MultiIndex([('0100', 3),
            ('0101', 2),
            ('0101', 3),
            ('0102', 2),
            ('0102', 3),
            ('0103', 2),
            ('0103', 3),
            ('0105', 3),
            ('0106', 2),
            ('0106', 3),
            ...
            ('5218', 3),
            ('5219', 2),
            ('5219', 3),
            ('5220', 2),
            ('5220', 3),
            ('5220', 5),
            ('5299', 3),
            ('5299', 5),
            ('5401', 3),
            ('5401', 5)],
           names=['code', 'credential_level'], length=590)

In [71]:
# error_df=error_df[error_df["4_yr_working_count"] >= 20]


error_df = (
    error_df
    .set_index(["code", "credential_level"])
    .loc[valid_codes]
    .reset_index()
)

error_df["school_count"] = (
    error_df.groupby(["code", "credential_level"])["school_name"]
    .transform("nunique")
)

error_df["confidence"] = pd.cut(
    error_df["school_count"],
    bins=[0, 5, 15, 100],
    labels=["low", "medium", "high"]
)

In [72]:
error_df.shape

(36219, 45)

In [73]:
error_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36219 entries, 0 to 36218
Data columns (total 45 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   code                            36219 non-null  object  
 1   credential_level                36219 non-null  int64   
 2   unit_id                         36219 non-null  string  
 3   distance                        36219 non-null  object  
 4   school_type                     36219 non-null  object  
 5   5_yr_working_count              36219 non-null  float64 
 6   school_state                    36219 non-null  string  
 7   location_lat                    36219 non-null  float64 
 8   location_lon                    36219 non-null  float64 
 9   locale                          36219 non-null  object  
 10  carnegie_size_setting           36219 non-null  object  
 11  admission_rate_overall          25209 non-null  Float64 
 12  median_family_inco

In [74]:
years = ["1", "4", "5"]

for y in years:
    # percent error
    error_df[f"{y}_year_pct_error"] = (
        error_df[f"{y}_year_error"] / error_df[f"{y}_year_pred"]
    )

    # weight (you can keep using 4yr count OR match per year if you have it)
    k = np.percentile(np.log1p(error_df[f"{y}_yr_working_count"]), 75)

    weight = (
        np.log1p(error_df[f"{y}_yr_working_count"]) /
        np.log1p(error_df[f"{y}_yr_working_count"] + k)
    )

    # final score per year
    error_df[f"{y}_year_score"] = (
        error_df[f"{y}_year_pct_error"] * weight
    )

“The weighting scheme introduces only minor adjustments to ranking positions, suggesting that prediction error remains dominant while program size provides a secondary refinement.”

To avoid instability in groups with small sample sizes, a small constant (epsilon) was added to the standard deviation when computing the final score. This prevents artificially inflated scores caused by near-zero variance estimates, while still allowing all groups to be included in the analysis.

To account for differences in sample size, a soft penalization factor was applied using sqrt(n / (n + k)). This approach reduces the influence of groups with small sample sizes without excluding them entirely. As n increases, the penalty diminishes, allowing larger groups to retain their full weight while appropriately down-weighting less reliable estimates.

In [75]:
error_df["rank_1"] = error_df.groupby(["code","credential_level"])["1_year_score"].rank(ascending=False, method="min")
error_df["rank_4"] = error_df.groupby(["code","credential_level"])["4_year_score"].rank(ascending=False, method="min")
error_df["rank_5"] = error_df.groupby(["code","credential_level"])["5_year_score"].rank(ascending=False, method="min")

In [76]:
error_df["move_1_to_4"] = error_df["rank_4"] - error_df["rank_1"]
error_df["move_4_to_5"] = error_df["rank_5"] - error_df["rank_4"]
error_df["move_1_to_5"] = error_df["rank_5"] - error_df["rank_1"]

In [77]:
error_df["rank_std"] = error_df[["rank_1","rank_4","rank_5"]].std(axis=1)
group_size = error_df.groupby("code")["code"].transform("count")

error_df["rank_std_pct"] = error_df["rank_std"] / (group_size - 1)

In [78]:
error_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_error_log,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_year_error_log,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_year_error_log,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct
0,0100,3,110422,1,Public,21.0,CA,35.299513,-120.657311,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.181225,"Agriculture, General.",California Polytechnic State University-San Lu...,-0.048828,68350.0,71770.218750,-3420.218750,84412.0,11.343465,70696.679688,11.166154,13715.320313,0.177311,36,64786.0,11.078845,46794.277344,10.753516,17991.722656,0.325329,18.0,28,high,0.384486,0.357262,0.194002,0.187573,-0.047655,-0.044850,2.0,2.0,14.0,0.0,12.0,12.0,6.928203,0.256600
1,0100,3,130934,1,Public,24.0,DE,39.187173,-75.540530,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.906647,"Agriculture, General.",Delaware State University,-0.138625,47478.0,54537.656250,-7059.656250,52676.0,10.871915,57368.062500,10.957243,-4692.062500,-0.085328,24,38873.0,10.568055,38589.582031,10.560738,283.417969,0.007318,22.0,28,high,0.007344,0.006927,-0.081789,-0.077499,-0.129446,-0.122880,16.0,15.0,19.0,-1.0,4.0,3.0,2.081666,0.077099
2,0100,3,145813,1,Public,132.0,IL,40.509403,-88.990058,22,16.0,0.8815,67099.0,0.48127,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.951447,"Agriculture, General.",Illinois State University,0.115831,64041.0,57036.546875,7004.453125,63600.0,11.060369,57882.054688,10.966163,5717.945313,0.094206,214,47295.0,10.764160,41597.433594,10.635794,5697.566406,0.128366,205.0,28,high,0.136969,0.136382,0.098786,0.098375,0.122806,0.121942,7.0,4.0,3.0,-3.0,-1.0,-4.0,2.081666,0.077099
3,0100,3,149222,1,Public,23.0,IL,37.714193,-89.217273,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.852016,"Agriculture, General.",Southern Illinois University-Carbondale,0.199365,63031.0,51638.171875,11392.828125,57596.0,10.961208,55643.027344,10.926712,1952.972656,0.034496,47,39700.0,10.589106,39304.937500,10.579105,395.062500,0.010001,22.0,28,high,0.010051,0.009480,0.035098,0.034243,0.220628,0.208893,15.0,6.0,1.0,-9.0,-5.0,-14.0,7.094599,0.262763
4,0100,3,149772,1,Public,145.0,IL,40.468086,-90.686899,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.898309,"Agriculture, General.",Western Illinois University,0.073401,58204.0,54084.820312,4119.179687,58333.0,10.973923,54959.113281,10.914345,3373.886719,0.059578,160,48509.0,10.789505,38580.050781,10.560491,9928.949219,0.229014,149.0,28,high,0.257360,0.255759,0.061389,0.061031,0.076161,0.075681,4.0,5.0,8.0,1.0,3.0,4.0,2.081666,0.077099


In [79]:
error_df.shape

(36219, 59)

In [80]:
error_df["rank_std_pct"].describe()

count    36219.000000
mean         0.115920
std          0.101417
min          0.000000
25%          0.037176
50%          0.085533
75%          0.168830
max          0.577350
Name: rank_std_pct, dtype: float64

When ranking schools within the same program, we observe an average rank movement of about 11%, indicating that performance is not stable over time even within comparable fields.

In [81]:
import plotly.express as px

px.histogram(error_df["rank_std_pct"])

In [82]:
error_df[["rank_std_pct", "4_yr_working_count"]].corr()

,rank_std_pct,4_yr_working_count
rank_std_pct,1.000000,-0.104228
4_yr_working_count,-0.104228,1.000000


In [83]:
top = error_df.nsmallest(100, "rank_4")   # best schools
bottom = error_df.nlargest(100, "rank_4")  # worst schools

print('top 100 std:',top["rank_std_pct"].mean())
print('bottom 100 std:',bottom["rank_std_pct"].mean())

top 100 std: 0.08244332635418383
bottom 100 std: 0.07545625425375754


We tested whether ranking instability was due to small sample sizes, but found no meaningful relationship. Even top-performing schools show similar or increased volatility, suggesting the instability is inherent to the earnings metric itself rather than noise.

In [84]:
program_stability = (
    error_df
    .groupby(["code", "credential_level"])
    .agg(
        mean_rank_std_pct=("rank_std_pct", "mean"),
        n=("school_name", "nunique")
    )
    .reset_index()
)

stable_programs = program_stability[
    program_stability["mean_rank_std_pct"] < 2   #Used to filter for stable programs
]
stable_df = error_df.merge(
    stable_programs[["code", "credential_level",'mean_rank_std_pct']],
    on=["code", "credential_level"],
    how="inner"
)
len(stable_df) / len(error_df)

1.0

In [85]:
stable_df.head()

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_error_log,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_year_error_log,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_year_error_log,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct
0,0100,3,110422,1,Public,21.0,CA,35.299513,-120.657311,23,16.0,0.3132,74513.0,0.455561,2.0,20.0,1,NaN,NaN,1036.0,mid,11.132397,11.181225,"Agriculture, General.",California Polytechnic State University-San Lu...,-0.048828,68350.0,71770.218750,-3420.218750,84412.0,11.343465,70696.679688,11.166154,13715.320313,0.177311,36,64786.0,11.078845,46794.277344,10.753516,17991.722656,0.325329,18.0,28,high,0.384486,0.357262,0.194002,0.187573,-0.047655,-0.044850,2.0,2.0,14.0,0.0,12.0,12.0,6.928203,0.256600,0.191572
1,0100,3,130934,1,Public,24.0,DE,39.187173,-75.540530,13,14.0,0.4658,39554.0,0.685488,2.0,20.0,1,940.0,NaN,746.0,mid,10.768022,10.906647,"Agriculture, General.",Delaware State University,-0.138625,47478.0,54537.656250,-7059.656250,52676.0,10.871915,57368.062500,10.957243,-4692.062500,-0.085328,24,38873.0,10.568055,38589.582031,10.560738,283.417969,0.007318,22.0,28,high,0.007344,0.006927,-0.081789,-0.077499,-0.129446,-0.122880,16.0,15.0,19.0,-1.0,4.0,3.0,2.081666,0.077099,0.191572
2,0100,3,145813,1,Public,132.0,IL,40.509403,-88.990058,22,16.0,0.8815,67099.0,0.48127,2.0,20.0,1,1113.0,24.0,2439.0,open,11.067279,10.951447,"Agriculture, General.",Illinois State University,0.115831,64041.0,57036.546875,7004.453125,63600.0,11.060369,57882.054688,10.966163,5717.945313,0.094206,214,47295.0,10.764160,41597.433594,10.635794,5697.566406,0.128366,205.0,28,high,0.136969,0.136382,0.098786,0.098375,0.122806,0.121942,7.0,4.0,3.0,-3.0,-1.0,-4.0,2.081666,0.077099,0.191572
3,0100,3,149222,1,Public,23.0,IL,37.714193,-89.217273,32,14.0,0.8688,37454.0,0.655376,2.0,22.0,1,1055.0,24.0,3237.0,open,11.051382,10.852016,"Agriculture, General.",Southern Illinois University-Carbondale,0.199365,63031.0,51638.171875,11392.828125,57596.0,10.961208,55643.027344,10.926712,1952.972656,0.034496,47,39700.0,10.589106,39304.937500,10.579105,395.062500,0.010001,22.0,28,high,0.010051,0.009480,0.035098,0.034243,0.220628,0.208893,15.0,6.0,1.0,-9.0,-5.0,-14.0,7.094599,0.262763,0.191572
4,0100,3,149772,1,Public,145.0,IL,40.468086,-90.686899,33,13.0,0.7118,36222.0,0.684009,2.0,21.0,1,NaN,NaN,1807.0,open,10.971709,10.898309,"Agriculture, General.",Western Illinois University,0.073401,58204.0,54084.820312,4119.179687,58333.0,10.973923,54959.113281,10.914345,3373.886719,0.059578,160,48509.0,10.789505,38580.050781,10.560491,9928.949219,0.229014,149.0,28,high,0.257360,0.255759,0.061389,0.061031,0.076161,0.075681,4.0,5.0,8.0,1.0,3.0,4.0,2.081666,0.077099,0.191572


In [86]:
stable_programs.shape

(590, 4)

In [87]:
stable_gap_df = stable_df.copy()

stable_gap_df["median_score"] = stable_gap_df[
    ["1_year_score", "4_year_score", "5_year_score"]
].median(axis=1)

program_gap = (
    stable_gap_df
    .groupby(["code", "credential_level"])
    .agg(
        min_score=("median_score", "min"),
        q1_score=("median_score", lambda x: x.quantile(0.25)),
        median_program_score=("median_score", "median"),
        q3_score=("median_score", lambda x: x.quantile(0.75)),
        max_score=("median_score", "max"),
        mean_score=("median_score", "mean"),
        std_score=("median_score", "std"),
        n=("school_name", "nunique")
    )
    .reset_index()
)
# raw gap: biggest under/over performer spread 

program_gap["gap"] = program_gap["max_score"] - program_gap["min_score"]

# robust gap: less sensitive to one weird school 
program_gap["iqr_gap"] = program_gap["q3_score"] - program_gap["q1_score"]

#  weight so tiny groups don't dominate 
program_gap["gap_weighted"] = (
    program_gap["gap"] * (program_gap["n"] / (program_gap["n"] + 10))
)
program_gap["iqr_gap_weighted"] = (
    program_gap["iqr_gap"] * (program_gap["n"] / (program_gap["n"] + 10))
)

#  bring back titles 
titles = error_df[["code", "credential_level", "title"]].drop_duplicates()

program_gap = program_gap.merge(
    titles,
    on=["code", "credential_level"],
    how="left"
)

# top programs with biggest variability 
top_gap_programs = (
    program_gap
    .sort_values("gap_weighted", ascending=False)
)

top_iqr_programs = (
    program_gap
    .sort_values("iqr_gap_weighted", ascending=False)
)

# views
top_gap_programs.head(20)

,code,credential_level,min_score,q1_score,median_program_score,q3_score,max_score,mean_score,std_score,n,gap,iqr_gap,gap_weighted,iqr_gap_weighted,title
480,5112,7,-0.466098,-0.065185,-0.008368,0.094668,1.916622,0.050716,0.247719,123,2.382721,0.159853,2.203569,0.147834,Medicine.
303,3099,3,-0.396303,-0.120491,-0.039356,0.036762,1.674184,-0.012196,0.209513,145,2.070487,0.157254,1.936907,0.147108,"Multi/Interdisciplinary Studies, Other."
229,2201,7,-0.388552,-0.086744,-0.000486,0.122466,1.255737,0.040030,0.211278,164,1.644290,0.209211,1.549790,0.197187,Law.
518,5138,5,-0.318010,-0.053547,0.001669,0.054238,1.161475,0.035950,0.201929,337,1.479485,0.107785,1.436849,0.104679,"Registered Nursing, Nursing Administration, Nu..."
461,5107,1,-0.309005,-0.062851,-0.016285,0.037360,1.118344,-0.010479,0.102274,364,1.427350,0.100211,1.389185,0.097531,Health and Medical Administrative Services.
471,5109,3,-0.411198,-0.036267,0.077048,0.204584,0.953461,0.095953,0.220930,139,1.364659,0.240851,1.273072,0.224687,"Allied Health Diagnostic, Intervention, and Tr..."
65,1101,3,-0.298650,-0.063371,0.033826,0.137687,0.991330,0.040806,0.159706,351,1.289980,0.201058,1.254247,0.195488,"Computer and Information Sciences, General."
472,5109,5,-0.536626,0.007809,0.085552,0.192860,0.792836,0.077714,0.203647,148,1.329462,0.185051,1.245319,0.173339,"Allied Health Diagnostic, Intervention, and Tr..."
536,5202,5,-0.464458,-0.074788,0.009674,0.119663,0.770061,0.031069,0.161308,703,1.234519,0.194451,1.217205,0.191723,"Business Administration, Management and Operat..."
469,5109,1,-0.420204,-0.097417,0.006440,0.125648,0.862944,0.037944,0.199968,155,1.283148,0.223066,1.205382,0.209546,"Allied Health Diagnostic, Intervention, and Tr..."


In [88]:
top_iqr_programs.head(10)

,code,credential_level,min_score,q1_score,median_program_score,q3_score,max_score,mean_score,std_score,n,gap,iqr_gap,gap_weighted,iqr_gap_weighted,title
528,5199,5,-0.246037,-0.217476,-0.005214,0.684019,1.160624,0.215798,0.564067,10,1.406661,0.901495,0.703331,0.450747,Health Professions and Related Clinical Scienc...
468,5108,5,-0.092124,0.553470,1.122478,1.454078,1.918225,1.009101,0.756668,9,2.010349,0.900608,0.952271,0.426604,Allied Health and Medical Assisting Services.
230,2202,5,-0.378565,-0.022957,0.104973,0.365250,0.644717,0.146586,0.279212,24,1.023282,0.388207,0.722317,0.274028,Legal Research and Advanced Professional Studies.
80,1108,3,-0.322299,-0.148307,-0.124442,0.184694,1.004446,0.009071,0.258817,37,1.326745,0.333000,1.044459,0.262149,Computer Software and Media Applications.
239,2313,3,-0.341825,-0.182310,-0.034111,0.107891,0.262856,-0.029357,0.159332,82,0.604681,0.290201,0.538955,0.258657,Rhetoric and Composition/Writing Studies.
422,5003,3,-0.469062,-0.229371,-0.064895,0.102131,0.316477,-0.064041,0.208367,25,0.785539,0.331502,0.561099,0.236787,Dance.
551,5207,5,-0.272949,-0.170808,0.186002,0.657165,1.102368,0.300356,0.634142,4,1.375316,0.827974,0.392948,0.236564,Entrepreneurial and Small Business Operations.
532,5201,5,-0.165017,-0.010610,0.118102,0.263989,0.654943,0.141732,0.196012,61,0.819960,0.274599,0.704473,0.235923,"Business/Commerce, General."
283,3001,5,-0.294622,-0.012996,0.461601,0.612296,0.779871,0.316631,0.436906,6,1.074493,0.625292,0.402935,0.234485,Biological and Physical Sciences.
471,5109,3,-0.411198,-0.036267,0.077048,0.204584,0.953461,0.095953,0.220930,139,1.364659,0.240851,1.273072,0.224687,"Allied Health Diagnostic, Intervention, and Tr..."


In [89]:
print(
    program_stability[
        (program_stability["code"] == "4301") &
        (program_stability["credential_level"] == 1)
    ]
)

     code  credential_level  mean_rank_std_pct   n
346  4301                 1           0.012161  97


In [90]:
(
    stable_gap_df[(stable_gap_df["code"] == "4301")&(stable_gap_df["credential_level"]==1)]
    .sort_values(["credential_level", "median_score"], ascending=[True, False])
)

,code,credential_level,unit_id,distance,school_type,5_yr_working_count,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket,5_year_earning_log,5_year_pred_log,title,school_name,5_year_error_log,5_year_earning,5_year_pred,5_year_error,4_year_earning,4_year_earning_log,4_year_pred,4_year_pred_log,4_year_error,4_year_error_log,4_yr_working_count,1_year_earning,1_year_earning_log,1_year_pred,1_year_pred_log,1_year_error,1_year_error_log,1_yr_working_count,school_count,confidence,1_year_pct_error,1_year_score,4_year_pct_error,4_year_score,5_year_pct_error,5_year_score,rank_1,rank_4,rank_5,move_1_to_4,move_4_to_5,move_1_to_5,rank_std,rank_std_pct,mean_rank_std_pct,median_score
16289,4301,1,123013,1,Public,105.0,CA,38.457090,-122.718974,12,4.0,<NA>,19251.0,0.977082,1.0,24.0,1,NaN,NaN,NaN,Missing,11.444689,10.992964,Criminal Justice and Corrections.,Santa Rosa Junior College,0.451726,93404.0,59454.335938,33949.664062,106298.0,11.574002,56006.324219,10.933220,50291.675781,0.640782,76,111649.0,11.623115,41424.000000,10.631616,70225.000000,0.991500,54.0,97,high,1.695273,1.660742,0.897964,0.885450,0.571021,0.565767,1.0,1.0,6.0,0.0,5.0,5.0,2.886751,0.002827,0.012161,0.885450
16290,4301,1,123527,1,Public,19.0,CA,34.086353,-117.313248,12,4.0,<NA>,16948.0,<NA>,1.0,25.0,1,NaN,NaN,NaN,Missing,10.724588,10.820326,Criminal Justice and Corrections.,San Bernardino Valley College,-0.095738,45460.0,50027.382812,-4567.382812,99385.0,11.506756,53717.074219,10.891486,45667.925781,0.615270,25,94285.0,11.454077,47179.015625,10.761704,47105.984375,0.692373,45.0,97,high,0.998452,0.973324,0.850157,0.807560,-0.091298,-0.085298,2.0,2.0,81.0,0.0,79.0,79.0,45.610671,0.044673,0.012161,0.807560
16288,4301,1,110246,1,Public,26.0,CA,39.649782,-121.644831,42,4.0,<NA>,16880.0,0.965035,1.0,23.0,1,NaN,NaN,NaN,Missing,11.343062,10.783738,Criminal Justice and Corrections.,Butte College,0.559324,84378.0,48230.074219,36147.925781,82708.0,11.323072,53805.410156,10.893129,28902.589844,0.429942,86,73170.0,11.200541,38570.527344,10.560244,34599.472656,0.640297,32.0,97,high,0.897044,0.863638,0.537169,0.530690,0.749489,0.714746,3.0,5.0,2.0,2.0,-3.0,-1.0,1.527525,0.001496,0.012161,0.714746
16359,4301,1,226134,1,Public,50.0,TX,27.506477,-99.520772,11,12.0,<NA>,17884.0,<NA>,1.0,21.0,1,NaN,NaN,NaN,Missing,11.282871,10.476936,Criminal Justice and Corrections.,Laredo College,0.805934,79449.0,35487.515625,43961.484375,77575.0,11.259000,54989.367188,10.914895,22585.632812,0.344105,43,72049.0,11.185102,41530.246094,10.634177,30518.753906,0.550925,25.0,97,high,0.734856,0.698671,0.410727,0.399630,1.238787,1.211669,5.0,13.0,1.0,8.0,-12.0,-4.0,6.110101,0.005984,0.012161,0.698671
16312,4301,1,136358,1,Public,139.0,FL,26.612560,-80.086530,21,15.0,<NA>,21408.0,0.921804,1.0,23.0,1,NaN,NaN,NaN,Missing,11.288531,10.774440,Criminal Justice and Corrections.,Palm Beach State College,0.514091,79900.0,47783.699219,32116.300781,78216.0,11.267230,54058.671875,10.897825,24157.328125,0.369404,153,69415.0,11.147858,38845.910156,10.567358,30569.089844,0.580500,168.0,97,high,0.786932,0.782677,0.446872,0.444124,0.672118,0.667665,4.0,10.0,3.0,6.0,-7.0,-1.0,3.785939,0.003708,0.012161,0.667665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16333,4301,1,184205,1,Public,31.0,NJ,39.440537,-75.057829,13,2.0,<NA>,35201.0,0.682554,1.0,22.0,1,NaN,NaN,NaN,Missing,10.497367,10.824514,Criminal Justice and Corrections.,Rowan College of South Jersey-Cumberland Campus,-0.327148,36220.0,50237.367188,-14017.367188,44362.0,10.700139,55877.375000,10.930915,-11515.375000,-0.230776,57,38144.0,10.549124,448

In [91]:
import plotly.express as px
bar_df = stable_gap_df[
    (stable_gap_df["code"] == "4301") & (stable_gap_df["credential_level"] == 1)
]

bar_df = bar_df.sort_values(by="median_score", ascending=True)

px.bar(
    bar_df,
    x="median_score",
    y="school_name",
    orientation="h",
    labels={"median_score":"3 year Pred. Error Average","school_name":"Schools"},
    title="Certificate in Criminal Justice and Corrections."
)

Programs such as Criminal Justice show substantial variability in outcomes across institutions, even after controlling for observable factors. This suggests that institutional effects such as program quality, networking opportunities, or industry connections may play a significant role in shaping student outcomes.

In [92]:
stable_gap_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36219 entries, 0 to 36218
Data columns (total 61 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   code                            36219 non-null  object  
 1   credential_level                36219 non-null  int64   
 2   unit_id                         36219 non-null  string  
 3   distance                        36219 non-null  object  
 4   school_type                     36219 non-null  object  
 5   5_yr_working_count              36219 non-null  float64 
 6   school_state                    36219 non-null  string  
 7   location_lat                    36219 non-null  float64 
 8   location_lon                    36219 non-null  float64 
 9   locale                          36219 non-null  object  
 10  carnegie_size_setting           36219 non-null  object  
 11  admission_rate_overall          25209 non-null  Float64 
 12  median_family_inco

In [93]:
save(stable_gap_df,file_name=f"{model_name}_national_residual_programs")
print(f'saved {model_name}_national_residual_programs.csv')

saved xgb_national_residual_programs.csv
